## Install Library

In [ ]:
# Install the YOLOv8 package
!pip install ultralytics

# Clear the output and check the GPU status
import ultralytics
ultralytics.checks()

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 0.1/235.7 GB disk)


## Mount with Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Test Before Tuned

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. Base directory
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"

# Load class metadata (names and class count) from baseline configuration
base_yaml_path = f"{base_dir}/Dataset_1_Baseline2/data_baseline2.yaml"
with open(base_yaml_path, 'r') as f:
    base_config = yaml.safe_load(f)

# Models and their respective dataset directories
models_to_test = {
    "Baseline2": {
        "ds_path": f"{base_dir}/Dataset_1_Baseline2",
        "weights": f"{base_dir}/Training_Results/Optimized_Model_Baseline2_YOLOv8n/weights/best.pt"
    },
    "CLAHE": {
        "ds_path": f"{base_dir}/Dataset_2_CLAHE",
        "weights": f"{base_dir}/Training_Results/Optimized_Model_CLAHE_YOLOv8n/weights/best.pt"
    },
    "AGCWD": {
        "ds_path": f"{base_dir}/Dataset_3_AGCWD",
        "weights": f"{base_dir}/Training_Results/Optimized_Model_AGCWD_YOLOv8n/weights/best.pt"
    },
    "SSR": {
        "ds_path": f"{base_dir}/Dataset_4_SSR",
        "weights": f"{base_dir}/Training_Results/Optimized_Model_SSR_YOLOv8n/weights/best.pt"
    }
}

# 2. Sequential Evaluation Loop
print("Starting Automated Performance Matrix Evaluation...\n")

for algo_name, config in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING MODEL: {algo_name}")
    print(f"{'='*60}")

    ds_path = config["ds_path"]
    model_path = config["weights"]

    # Verify model weights exist before testing
    if not os.path.exists(model_path):
        print(f"Warning: Weights not found at {model_path}. Skipping {algo_name}...")
        continue

    model = YOLO(model_path)

    # Define test conditions
    day_img_path = os.path.join(ds_path, 'dataset_split', 'test_day', 'images')
    night_img_path = os.path.join(ds_path, 'dataset_split', 'test_night', 'images')

    conditions = {
        "Day": day_img_path,
        "Night": night_img_path,
        "Combined": [day_img_path, night_img_path]
    }

    for condition_name, test_paths in conditions.items():
        print(f"\nRunning {algo_name} on [{condition_name}] set...")

        # Create a dynamic evaluation YAML
        eval_yaml_path = os.path.join(ds_path, f"eval_{condition_name.lower()}.yaml")

        # Added a dummy 'train' path to satisfy YOLOv8's syntax checker
        eval_config = {
            'names': base_config.get('names', {0: 'pothole'}),
            'nc': base_config.get('nc', 1),
            'train': os.path.join(ds_path, 'train', 'images'),
            'val': test_paths
        }

        with open(eval_yaml_path, 'w') as f:
            yaml.dump(eval_config, f)

        # Run YOLO validation
        model.val(
            data=eval_yaml_path,
            split='val',
            project=f"{base_dir}/Matrix_Test_Results",
            name=f"Matrix_Eval_{algo_name}_{condition_name}",
            exist_ok=True,
            device='0'
        )

print("\nALL EVALUATIONS COMPLETED! Results saved to /Test_Results.")

Starting Automated Performance Matrix Evaluation...


EVALUATING MODEL: Baseline2

Running Baseline2 on [Day] set...
Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 7.6±15.1 ms, read: 8.8±19.4 MB/s, size: 67.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/dataset_split/test_day/labels... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 2.6it/s 1:53
val: New cache created: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/dataset_split/test_day/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 3.7it/s 5.1s
                   all        294       

## Test After Tuned

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. Base directory
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"

# Load class metadata (names and class count) from baseline configuration
base_yaml_path = f"{base_dir}/Dataset_1_Baseline2/data_baseline2.yaml"
with open(base_yaml_path, 'r') as f:
    base_config = yaml.safe_load(f)

# Models and their respective dataset directories
models_to_test = {
    "Baseline2": {
        "ds_path": f"{base_dir}/Dataset_1_Baseline2",
        "weights": f"{base_dir}/Training_Results/Optimized_Model_Baseline2_YOLOv8n/weights/best.pt"
    },
    "CLAHE": {
        "ds_path": f"{base_dir}/Dataset_2_CLAHE_Tuned",
        "weights": f"{base_dir}/Training_Results/Tuned_Model_CLAHE_YOLOv8n/weights/best.pt"
    },
    "AGCWD": {
        "ds_path": f"{base_dir}/Dataset_3_AGCWD_Tuned",
        "weights": f"{base_dir}/Training_Results/Tuned_Model_AGCWD_YOLOv8n/weights/best.pt"
    },
    "SSR": {
        "ds_path": f"{base_dir}/Dataset_4_SSR_Tuned",
        "weights": f"{base_dir}/Training_Results/Tuned_Model_SSR_YOLOv8n/weights/best.pt"
    }
}

# 2. Sequential Evaluation Loop
print("Starting Automated Performance Tuned Matrix Evaluation...\n")

for algo_name, config in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING MODEL: {algo_name}")
    print(f"{'='*60}")

    ds_path = config["ds_path"]
    model_path = config["weights"]

    # Verify model weights exist before testing
    if not os.path.exists(model_path):
        print(f"Warning: Weights not found at {model_path}. Skipping {algo_name}...")
        continue

    model = YOLO(model_path)

    # Define test conditions
    day_img_path = os.path.join(ds_path, 'dataset_split', 'test_day', 'images')
    night_img_path = os.path.join(ds_path, 'dataset_split', 'test_night', 'images')

    conditions = {
        "Day": day_img_path,
        "Night": night_img_path,
        "Combined": [day_img_path, night_img_path]
    }

    for condition_name, test_paths in conditions.items():
        print(f"\nRunning {algo_name} on [{condition_name}] set...")

        # Create a dynamic evaluation YAML
        eval_yaml_path = os.path.join(ds_path, f"eval_{condition_name.lower()}.yaml")

        # Added a dummy 'train' path to satisfy YOLOv8's syntax checker
        eval_config = {
            'names': base_config.get('names', {0: 'pothole'}),
            'nc': base_config.get('nc', 1),
            'train': os.path.join(ds_path, 'train', 'images'),
            'val': test_paths
        }

        with open(eval_yaml_path, 'w') as f:
            yaml.dump(eval_config, f)

        # Run YOLO validation
        model.val(
            data=eval_yaml_path,
            split='val',
            project=f"{base_dir}/Tuned_Matrix_Test_Results",
            name=f"Tuned_Matrix_Eval_{algo_name}_{condition_name}",
            exist_ok=True,
            device='0'
        )

print("\nALL EVALUATIONS COMPLETED! Results saved to /Test_Results.")

Starting Automated Performance Tuned Matrix Evaluation...


EVALUATING MODEL: Baseline2

Running Baseline2 on [Day] set...
Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 2.3±3.0 ms, read: 8.9±19.7 MB/s, size: 60.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/dataset_split/test_day/labels... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 3.3it/s 1:28
val: New cache created: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/dataset_split/test_day/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 3.7it/s 5.2s
                   all        294  

## Second Tuned

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. Base directory
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"

# Load class metadata (names and class count) from baseline configuration
base_yaml_path = f"{base_dir}/Dataset_1_Baseline2/data_baseline2.yaml"
with open(base_yaml_path, 'r') as f:
    base_config = yaml.safe_load(f)

# Models and their respective dataset directories
models_to_test = {
    "Baseline2": {
        "ds_path": f"{base_dir}/Dataset_1_Baseline2",
        "weights": f"{base_dir}/Training_Results/Optimized_Model_Baseline2_YOLOv8n/weights/best.pt"
    },
    "CLAHE": {
        "ds_path": f"{base_dir}/Dataset_2_CLAHE_Tuned(1.5, [8,8])",
        "weights": f"{base_dir}/Training_Results/Tuned(1.5, [8,8])_Model_CLAHE_YOLOv8n/weights/best.pt"
    },
    "AGCWD": {
        "ds_path": f"{base_dir}/Dataset_3_AGCWD_Tuned(0.3)",
        "weights": f"{base_dir}/Training_Results/Tuned(0.3)_Model_AGCWD_YOLOv8n/weights/best.pt"
    },
    "SSR": {
        "ds_path": f"{base_dir}/Dataset_4_SSR_Tuned(80)",
        "weights": f"{base_dir}/Training_Results/Tuned(80)_Model_SSR_YOLOv8n/weights/best.pt"
    }
}

# 2. Sequential Evaluation Loop
print("Starting Automated Performance Tuned Matrix Evaluation...\n")

for algo_name, config in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING MODEL: {algo_name}")
    print(f"{'='*60}")

    ds_path = config["ds_path"]
    model_path = config["weights"]

    # Verify model weights exist before testing
    if not os.path.exists(model_path):
        print(f"Warning: Weights not found at {model_path}. Skipping {algo_name}...")
        continue

    model = YOLO(model_path)

    # Define test conditions
    day_img_path = os.path.join(ds_path, 'dataset_split', 'test_day', 'images')
    night_img_path = os.path.join(ds_path, 'dataset_split', 'test_night', 'images')

    conditions = {
        "Day": day_img_path,
        "Night": night_img_path,
        "Combined": [day_img_path, night_img_path]
    }

    for condition_name, test_paths in conditions.items():
        print(f"\nRunning {algo_name} on [{condition_name}] set...")

        # Create a dynamic evaluation YAML
        eval_yaml_path = os.path.join(ds_path, f"eval_{condition_name.lower()}.yaml")

        # Added a dummy 'train' path to satisfy YOLOv8's syntax checker
        eval_config = {
            'names': base_config.get('names', {0: 'pothole'}),
            'nc': base_config.get('nc', 1),
            'train': os.path.join(ds_path, 'train', 'images'),
            'val': test_paths
        }

        with open(eval_yaml_path, 'w') as f:
            yaml.dump(eval_config, f)

        # Run YOLO validation
        model.val(
            data=eval_yaml_path,
            split='val',
            project=f"{base_dir}/Tuned_2_Matrix_Test_Results",
            name=f"Tuned_2_Matrix_Eval_{algo_name}_{condition_name}",
            exist_ok=True,
            device='0'
        )

print("\nALL EVALUATIONS COMPLETED! Results saved to /Test_Results.")

Starting Automated Performance Tuned Matrix Evaluation...


EVALUATING MODEL: Baseline2

Running Baseline2 on [Day] set...
Ultralytics 8.4.198 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.9±3.1 ms, read: 14.9±32.8 MB/s, size: 92.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/dataset_split/test_day/labels... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 3.6it/s 1:23
val: New cache created: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/dataset_split/test_day/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 3.4it/s 5.6s
                   all        294 

## Mixed

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. Base directory
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"

# Load class metadata (names and class count) from baseline configuration
base_yaml_path = f"{base_dir}/Dataset_1_Baseline2/data_baseline2.yaml"
with open(base_yaml_path, 'r') as f:
    base_config = yaml.safe_load(f)

# Models and their respective dataset directories
models_to_test = {
    "Baseline2": {
        "ds_path": f"{base_dir}/Dataset_1_Baseline2",
        "weights": f"{base_dir}/Training_Results/Optimized_Model_Baseline2_YOLOv8n/weights/best.pt"
    },
    "CLAHE": {
        "ds_path": f"{base_dir}/Dataset_2_CLAHE_Mixed",
        "weights": f"{base_dir}/Training_Results/Mixed_Model_CLAHE_Mixed_YOLOv8n/weights/best.pt"
    },
    "AGCWD": {
        "ds_path": f"{base_dir}/Dataset_3_AGCWD_Mixed",
        "weights": f"{base_dir}/Training_Results/Mixed_Model_AGCWD_Mixed_YOLOv8n/weights/best.pt"
    },
    "SSR": {
        "ds_path": f"{base_dir}/Dataset_4_SSR_Mixed",
        "weights": f"{base_dir}/Training_Results/Mixed_Model_SSR_Mixed_YOLOv8n/weights/best.pt"
    }
}

# 2. Sequential Evaluation Loop
print("Starting Automated Performance Mixed Matrix Evaluation...\n")

for algo_name, config in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING MODEL: {algo_name}")
    print(f"{'='*60}")

    ds_path = config["ds_path"]
    model_path = config["weights"]

    # Verify model weights exist before testing
    if not os.path.exists(model_path):
        print(f"Warning: Weights not found at {model_path}. Skipping {algo_name}...")
        continue

    model = YOLO(model_path)

    # Define test conditions
    day_img_path = os.path.join(ds_path, 'dataset_split', 'test_day', 'images')
    night_img_path = os.path.join(ds_path, 'dataset_split', 'test_night', 'images')

    conditions = {
        "Day": day_img_path,
        "Night": night_img_path,
        "Combined": [day_img_path, night_img_path]
    }

    for condition_name, test_paths in conditions.items():
        print(f"\nRunning {algo_name} on [{condition_name}] set...")

        # Create a dynamic evaluation YAML
        eval_yaml_path = os.path.join(ds_path, f"eval_{condition_name.lower()}.yaml")

        # Added a dummy 'train' path to satisfy YOLOv8's syntax checker
        eval_config = {
            'names': base_config.get('names', {0: 'pothole'}),
            'nc': base_config.get('nc', 1),
            'train': os.path.join(ds_path, 'train', 'images'),
            'val': test_paths
        }

        with open(eval_yaml_path, 'w') as f:
            yaml.dump(eval_config, f)

        # Run YOLO validation
        model.val(
            data=eval_yaml_path,
            split='val',
            project=f"{base_dir}/Mixed_Matrix_Test_Results",
            name=f"Mixed_Matrix_Eval_{algo_name}_{condition_name}",
            exist_ok=True,
            device='0'
        )

print("\nALL EVALUATIONS COMPLETED! Results saved to /Test_Results.")

Starting Automated Performance Mixed Matrix Evaluation...


EVALUATING MODEL: Baseline2

Running Baseline2 on [Day] set...
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.3 ms, read: 45.0±24.7 MB/s, size: 72.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/dataset_split/test_day/labels... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 217.9it/s 1.3s
val: New cache created: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/dataset_split/test_day/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 4.2it/s 4.6s
                   all        29

## CLAHE Blend (0.7, 0.8)

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. Base directory
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"

# Load class metadata (names and class count) from baseline config
base_yaml_path = f"{base_dir}/Dataset_1_Baseline2/data_baseline2.yaml"
with open(base_yaml_path, 'r') as f:
    base_config = yaml.safe_load(f)

# Models and their respective dataset directories
models_to_test = {
    "CLAHE(0.7)": {
        "ds_path": f"{base_dir}/Dataset_2_CLAHE_Blend",
        "weights": f"{base_dir}/Training_Results/Blend_Model_CLAHE Blend_YOLOv8n/weights/best.pt"
    },
    "CLAHE(0.8)": {
        "ds_path": f"{base_dir}/Dataset_2_CLAHE_Blend(0.8)",
        "weights": f"{base_dir}/Training_Results/Blend(0.8)_Model_CLAHE Blend(0.8)_YOLOv8n/weights/best.pt"
    },
}

# 2. Sequential Evaluation Loop
print("Starting Automated Performance Blend Matrix Evaluation...\n")

for algo_name, config in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING MODEL: {algo_name}")
    print(f"{'='*60}")

    ds_path = config["ds_path"]
    model_path = config["weights"]

    # Verify model weights exist before testing
    if not os.path.exists(model_path):
        print(f"Warning: Weights not found at {model_path}. Skipping {algo_name}...")
        continue

    model = YOLO(model_path)

    # Define test conditions
    day_img_path = os.path.join(ds_path, 'dataset_split', 'test_day', 'images')
    night_img_path = os.path.join(ds_path, 'dataset_split', 'test_night', 'images')

    conditions = {
        "Day": day_img_path,
        "Night": night_img_path,
        "Combined": [day_img_path, night_img_path]
    }

    for condition_name, test_paths in conditions.items():
        print(f"\nRunning {algo_name} on [{condition_name}] set...")

        # Create a dynamic evaluation YAML
        eval_yaml_path = os.path.join(ds_path, f"eval_{condition_name.lower()}.yaml")

        # FIXED: Added a dummy 'train' path to satisfy YOLOv8's syntax checker
        eval_config = {
            'names': base_config.get('names', {0: 'pothole'}),
            'nc': base_config.get('nc', 1),
            'train': os.path.join(ds_path, 'train', 'images'),
            'val': test_paths
        }

        with open(eval_yaml_path, 'w') as f:
            yaml.dump(eval_config, f)

        # Run YOLO validation
        model.val(
            data=eval_yaml_path,
            split='val',
            project=f"{base_dir}/Blend_Matrix_Test_Results",
            name=f"Blend_Matrix_Eval_{algo_name}_{condition_name}",
            exist_ok=True,
            device='0'  # Enforce GPU evaluation
        )

print("\nALL EVALUATIONS COMPLETED! Results saved to /Test_Results.")

Starting Automated Performance Blend Matrix Evaluation...


EVALUATING MODEL: CLAHE(0.7)

Running CLAHE(0.7) on [Day] set...
Ultralytics 8.4.210 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.9±0.7 ms, read: 21.6±48.1 MB/s, size: 146.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend/dataset_split/test_day/labels... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 1.4it/s 3:34
val: New cache created: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend/dataset_split/test_day/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.3s
                   all     

# CLAHE Blend (0.6)

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. Base directory
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"

# Load class metadata (names and class count) from baseline configuration
base_yaml_path = f"{base_dir}/Dataset_1_Baseline2/data_baseline2.yaml"
with open(base_yaml_path, 'r') as f:
    base_config = yaml.safe_load(f)

# Models and their respective dataset directories
models_to_test = {
    "CLAHE(0.6)": {
        "ds_path": f"{base_dir}/Dataset_2_CLAHE_Blend(0.6)",
        "weights": f"{base_dir}/Training_Results/Blend(0.6)_Model_CLAHE_YOLOv8n/weights/best.pt"
    }
}

# 2. Sequential Evaluation Loop
print("Starting Automated Performance Blend Matrix Evaluation...\n")

for algo_name, config in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING MODEL: {algo_name}")
    print(f"{'='*60}")

    ds_path = config["ds_path"]
    model_path = config["weights"]

    # Verify model weights exist before testing
    if not os.path.exists(model_path):
        print(f"Warning: Weights not found at {model_path}. Skipping {algo_name}...")
        continue

    model = YOLO(model_path)

    # Define test conditions
    day_img_path = os.path.join(ds_path, 'dataset_split', 'test_day', 'images')
    night_img_path = os.path.join(ds_path, 'dataset_split', 'test_night', 'images')

    conditions = {
        "Day": day_img_path,
        "Night": night_img_path,
        "Combined": [day_img_path, night_img_path]
    }

    for condition_name, test_paths in conditions.items():
        print(f"\nRunning {algo_name} on [{condition_name}] set...")

        # Create a dynamic evaluation YAML
        eval_yaml_path = os.path.join(ds_path, f"eval_{condition_name.lower()}.yaml")

        # Added a dummy 'train' path to satisfy YOLOv8's syntax checker
        eval_config = {
            'names': base_config.get('names', {0: 'pothole'}),
            'nc': base_config.get('nc', 1),
            'train': os.path.join(ds_path, 'train', 'images'),
            'val': test_paths
        }

        with open(eval_yaml_path, 'w') as f:
            yaml.dump(eval_config, f)

        # Run YOLO validation
        model.val(
            data=eval_yaml_path,
            split='val',
            project=f"{base_dir}/Blend_Matrix_Test_Results",
            name=f"Blend_Matrix_Eval_{algo_name}_{condition_name}",
            exist_ok=True,
            device='0'
        )

print("\nALL EVALUATIONS COMPLETED! Results saved to /Test_Results.")

Starting Automated Performance Blend Matrix Evaluation...


EVALUATING MODEL: CLAHE(0.6)

Running CLAHE(0.6) on [Day] set...
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 17.1±36.8 ms, read: 4.1±8.6 MB/s, size: 154.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend(0.6)/dataset_split/test_day/labels... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 1.7it/s 2:58
val: New cache created: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend(0.6)/dataset_split/test_day/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.9it/s 6.6s
                 

## AGCWD Blend (0.6, 0.7, 0.8)

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. Base directory and paths setup
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"

# Load class metadata (names and class count) from baseline config
base_yaml_path = f"{base_dir}/Dataset_1_Baseline2/data_baseline2.yaml"
with open(base_yaml_path, 'r') as f:
    base_config = yaml.safe_load(f)

# Models and their respective dataset directories
models_to_test = {
    "AGCWD(0.6)": {
        "ds_path": f"{base_dir}/Dataset_3_AGCWD_Blend(0.6)",
        "weights": f"{base_dir}/Training_Results/Blend(0.6)_Model_AGCWD_YOLOv8n/weights/best.pt"
    },
    "AGCWD(0.7)": {
        "ds_path": f"{base_dir}/Dataset_3_AGCWD_Blend(0.7)",
        "weights": f"{base_dir}/Training_Results/Blend(0.7)_Model_AGCWD_YOLOv8n/weights/best.pt"
    },
    "AGCWD(0.8)": {
        "ds_path": f"{base_dir}/Dataset_3_AGCWD_Blend(0.8)",
        "weights": f"{base_dir}/Training_Results/Blend(0.8)_Model_AGCWD_YOLOv8n/weights/best.pt"
    },
}

# 2. Sequential Evaluation Loop
print("Starting Automated Performance Blend Matrix Evaluation...\n")

for algo_name, config in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING MODEL: {algo_name}")
    print(f"{'='*60}")

    ds_path = config["ds_path"]
    model_path = config["weights"]

    # Verify model weights exist before testing
    if not os.path.exists(model_path):
        print(f"Warning: Weights not found at {model_path}. Skipping {algo_name}...")
        continue

    model = YOLO(model_path)

    # Define test conditions
    day_img_path = os.path.join(ds_path, 'dataset_split', 'test_day', 'images')
    night_img_path = os.path.join(ds_path, 'dataset_split', 'test_night', 'images')

    conditions = {
        "Day": day_img_path,
        "Night": night_img_path,
        "Combined": [day_img_path, night_img_path]
    }

    for condition_name, test_paths in conditions.items():
        print(f"\nRunning {algo_name} on [{condition_name}] set...")

        # Create a dynamic evaluation YAML
        eval_yaml_path = os.path.join(ds_path, f"eval_{condition_name.lower()}.yaml")

        # FIXED: Added a dummy 'train' path to satisfy YOLOv8's syntax checker
        eval_config = {
            'names': base_config.get('names', {0: 'pothole'}),
            'nc': base_config.get('nc', 1),
            'train': os.path.join(ds_path, 'train', 'images'),
            'val': test_paths
        }

        with open(eval_yaml_path, 'w') as f:
            yaml.dump(eval_config, f)

        # Run YOLO validation
        model.val(
            data=eval_yaml_path,
            split='val',
            project=f"{base_dir}/Blend_Matrix_Test_Results",
            name=f"Blend_Matrix_Eval_{algo_name}_{condition_name}",
            exist_ok=True,
            device='0'  # Enforce GPU evaluation
        )

print("\nALL EVALUATIONS COMPLETED! Results saved to /Test_Results.")

Starting Automated Performance Blend Matrix Evaluation...


EVALUATING MODEL: AGCWD(0.6)

Running AGCWD(0.6) on [Day] set...
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 17.8±38.2 ms, read: 4.5±9.2 MB/s, size: 149.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_3_AGCWD_Blend(0.6)/dataset_split/test_day/labels... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 3.2it/s 1:32
val: New cache created: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_3_AGCWD_Blend(0.6)/dataset_split/test_day/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.9it/s 6.6s
                 

## AGCWD Blend (0.5)

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. Base directory and paths setup
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"

# Load class metadata (names and class count) from baseline config
base_yaml_path = f"{base_dir}/Dataset_1_Baseline2/data_baseline2.yaml"
with open(base_yaml_path, 'r') as f:
    base_config = yaml.safe_load(f)

# Models and their respective dataset directories
models_to_test = {
    "AGCWD(0.5)": {
        "ds_path": f"{base_dir}/Dataset_3_AGCWD_Blend(0.5)",
        "weights": f"{base_dir}/Training_Results/Blend(0.5)_Model_AGCWD_YOLOv8n/weights/best.pt"
    }
}

# 2. Sequential Evaluation Loop
print("Starting Automated Performance Blend Matrix Evaluation...\n")

for algo_name, config in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING MODEL: {algo_name}")
    print(f"{'='*60}")

    ds_path = config["ds_path"]
    model_path = config["weights"]

    # Verify model weights exist before testing
    if not os.path.exists(model_path):
        print(f"Warning: Weights not found at {model_path}. Skipping {algo_name}...")
        continue

    model = YOLO(model_path)

    # Define test conditions
    day_img_path = os.path.join(ds_path, 'dataset_split', 'test_day', 'images')
    night_img_path = os.path.join(ds_path, 'dataset_split', 'test_night', 'images')

    conditions = {
        "Day": day_img_path,
        "Night": night_img_path,
        "Combined": [day_img_path, night_img_path]
    }

    for condition_name, test_paths in conditions.items():
        print(f"\nRunning {algo_name} on [{condition_name}] set...")

        # Create a dynamic evaluation YAML
        eval_yaml_path = os.path.join(ds_path, f"eval_{condition_name.lower()}.yaml")

        # FIXED: Added a dummy 'train' path to satisfy YOLOv8's syntax checker
        eval_config = {
            'names': base_config.get('names', {0: 'pothole'}),
            'nc': base_config.get('nc', 1),
            'train': os.path.join(ds_path, 'train', 'images'),
            'val': test_paths
        }

        with open(eval_yaml_path, 'w') as f:
            yaml.dump(eval_config, f)

        # Run YOLO validation
        model.val(
            data=eval_yaml_path,
            split='val',
            project=f"{base_dir}/Blend_Matrix_Test_Results",
            name=f"Blend_Matrix_Eval_{algo_name}_{condition_name}",
            exist_ok=True,
            device='0'  # Enforce GPU evaluation
        )

print("\nALL EVALUATIONS COMPLETED! Results saved to /Test_Results.")

Starting Automated Performance Blend Matrix Evaluation...


EVALUATING MODEL: AGCWD(0.5)

Running AGCWD(0.5) on [Day] set...
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.3 ms, read: 79.7±30.1 MB/s, size: 132.7 KB)
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_3_AGCWD_Blend(0.5)/dataset_split/test_day/labels... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 2.2it/s 2:12
val: New cache created: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_3_AGCWD_Blend(0.5)/dataset_split/test_day/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.7it/s 11.1s
                   all        294        594      0.866      0.813      0.877      0.591
Speed: 1.7ms preprocess, 5.2ms inference, 0.0ms loss, 2.2ms postprocess per image
